In [ ]:
import os

# Set this environment variable BEFORE importing numpy/sklearn if possible,
# or at least before running the compute-heavy task.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"  # Often helpful to restrict OMP as well


import scanpy as sc
import numpy as np
import pandas as pd
import plotnine as gg
from data_resources import load_crispri_data, load_fitness_data

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.manifold import TSNE

In [ ]:
def vis_spacers(adata, gene, color="spacer"):
    adata_sub_onegene = adata[adata.obs["gene"] == gene].copy()

    return (
        gg.ggplot(
            adata.obs,
            gg.aes(x="UMAP1", y="UMAP2"),
        )
        + gg.geom_point(size=0.5, stroke=0)
        + gg.geom_point(adata_sub_onegene.obs, gg.aes(x="UMAP1", y="UMAP2", color=color))
    )

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
timeseries_df = pd.read_pickle(
    "/workspace/data/Eaton_2025/Data/lDE20_Imaging/Clustering/2023-01-23_sgRNA_Timeseries_df.pkl"
)
timeseries_df.info()

adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
fitness_df = load_fitness_data()

ops_df = pd.read_csv(
    os.path.join(
        "/workspace/data/Eaton_2025/Data/lDE20_Imaging/2024-01-25_lDE20_Steady_State_df_Estimators_wStats.csv"
    )
)

In [ ]:
spacer_fitness = fitness_df.loc[:, ["T1", "T2", "T3", "T4"]]
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
spacer_pca = pca.fit_transform(spacer_fitness)
fitness_df["FITNESS_PC1"] = spacer_pca[:, 0]
fitness_df["FITNESS_PC2"] = spacer_pca[:, 1]

In [ ]:
# TSNE_fitness = TSNE(n_components=2)
# embeddings_tsne = TSNE_fitness.fit_transform(spacer_fitness)
# fitness_df["TSNE1"] = embeddings_tsne[:, 0]
# fitness_df["TSNE2"] = embeddings_tsne[:, 1]
# (gg.ggplot(fitness_df, gg.aes(x="TSNE1", y="TSNE2", color="T4")) + gg.geom_point())

In [ ]:
(gg.ggplot(fitness_df, gg.aes(x="FITNESS_PC1", y="FITNESS_PC2", color="T4")) + gg.geom_point())

In [ ]:
adata.obs["UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["UMAP2"] = adata.obsm["X_umap"][:, 1]

(gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2", color="T4")) + gg.geom_point(size=0.5))

In [ ]:
# sc.tl.leiden(adata, resolution=1.0, key_added="leiden_1.0")
# sc.tl.leiden(adata, resolution=0.5, key_added="leiden_0.5")
# sc.tl.leiden(adata, resolution=0.25, key_added="leiden_0.25")
# sc.tl.leiden(adata, resolution=0.1, key_added="leiden_0.1")

In [ ]:
(gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2", color="leiden_0.25")) + gg.geom_point(size=0.5))

In [ ]:
# mapper = {
#     "0": "control-like",
#     "1": "control-like",
#     "2": "case-like",
#     "3": "case-like",
#     "4": "case-like",
#     "5": "case-like",
# }

# adata.obs["annotated_cluster"] = adata.obs["leiden_0.25"].map(mapper)

(
    gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2", color="annotated_cluster"))
    + gg.geom_point(size=0.5)
)

In [ ]:
predictability_score_ = (
    adata.obs.groupby("spacer")["annotated_cluster"]
    .value_counts(normalize=True)
    .to_frame("predictability_score")
    .reset_index()
    .query("annotated_cluster == 'case-like'")
)
predictability_score_

In [ ]:
predictability_score = predictability_score_.merge(
    fitness_df, left_on="spacer", right_index=True, how="left"
).assign(late_effect_score=lambda x: (x["T2"] / x["T4"]))

## comparison with OPS

In [ ]:
working_dir = "/workspace/data/Eaton_2025/Data/lDE20_Imaging"
df = pd.read_csv(
    os.path.join(working_dir, "2024-01-25_lDE20_Steady_State_df_Estimators_wStats.csv")
)

df_ = df.query("Estimator == 'Mean (Robust)'")
ops_summarystats_df = (
    df_.pivot(index=["sgRNA", "Gene", "N Mismatch"], columns="Variable(s)", values="Value")
    .reset_index()
    .set_index(["sgRNA", "Gene"])
)
ops_summarystats_df

In [ ]:
timeseries_df = pd.read_pickle(
    "/workspace/data/Eaton_2025/Data/lDE20_Imaging/Clustering/2023-01-23_sgRNA_Timeseries_df.pkl"
)
timeseries_df.info()


def expand_embeddings(df, columns):
    expanded_dfs = []
    for col in columns:
        if col not in df.columns:
            continue
        col_values = np.stack(df[col].values)
        if col_values.ndim > 2:
            col_values = col_values.reshape(len(df), -1)

        expanded = pd.DataFrame(
            col_values, index=df.index, columns=[f"{col}_{i}" for i in range(col_values.shape[1])]
        )
        expanded_dfs.append(expanded)

    metadata_cols = [c for c in df.columns if c not in columns]
    return pd.concat([df[metadata_cols]] + expanded_dfs, axis=1).reset_index()


df_embeddings = expand_embeddings(timeseries_df, ["Feature Vector"])
tsne_rep = TSNE(n_components=2)
embeddings_tsne = tsne_rep.fit_transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["tsne_x"] = embeddings_tsne[:, 0]
df_embeddings["tsne_y"] = embeddings_tsne[:, 1]
df_embeddings.info()

pca_ = PCA(n_components=2)
pca_.fit(df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]])
embeddings_pca = pca_.transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["pca_x"] = embeddings_pca[:, 0]
df_embeddings["pca_y"] = embeddings_pca[:, 1]
# comparison with growth fitness screen

In [ ]:
df_embeddings_ = df_embeddings.merge(ops_summarystats_df.reset_index(), on="sgRNA", how="left")
df_embeddings_

In [ ]:
(
    gg.ggplot(
        df_embeddings_,
        gg.aes(x="tsne_x", y="tsne_y", color="mCherry mean_intensity"),
    )
    + gg.geom_point(size=0.5)
)

In [ ]:
(
    gg.ggplot(
        df_embeddings_,
        gg.aes(x="tsne_x", y="tsne_y", color="Delta time (s)"),
    )
    + gg.geom_point(size=0.5)
)

In [ ]:
(
    gg.ggplot(
        df_embeddings_,
        gg.aes(x="tsne_x", y="tsne_y", color="Instantaneous Growth Rate: Volume"),
    )
    + gg.geom_point(size=0.5)
    + gg.labs(color="volumic growth rate")
)

In [ ]:
predictability_score_with_ops = predictability_score.merge(
    df_embeddings_, left_on="spacer", right_on="sgRNA", how="left"
)

In [ ]:
(
    gg.ggplot(
        predictability_score_with_ops.query("T4 <= -3"),
        gg.aes(x="Delta time (s)", y="T4", color="predictability_score"),
    )
    + gg.geom_point(size=0.5)
    + gg.geom_point(gg.aes(color="predictability_score"))
    + gg.theme_minimal()
    + gg.theme(figure_size=(5, 3))
)

In [ ]:
(
    gg.ggplot(
        predictability_score_with_ops.query("T4 <= -3"),
        gg.aes(x="Instantaneous Growth Rate: Volume", y="T4", color="predictability_score"),
    )
    + gg.geom_point(size=0.5)
    + gg.geom_point(gg.aes(color="predictability_score"))
    + gg.theme_minimal()
    + gg.theme(figure_size=(5, 3))
)

In [ ]:
(
    gg.ggplot(
        predictability_score_with_ops.query("T4 <= -3"),
        gg.aes(x="mCherry mean_intensity", y="T4", color="predictability_score"),
    )
    + gg.geom_point(size=0.5)
    + gg.geom_point(gg.aes(color="predictability_score"))
    + gg.theme_minimal()
    + gg.theme(figure_size=(5, 3))
)

In [ ]:
(
    gg.ggplot(
        predictability_score_with_ops,
        gg.aes(x="tsne_x", y="tsne_y"),
    )
    + gg.geom_point(size=0.5, data=df_embeddings)
    + gg.geom_point(gg.aes(color="predictability_score"))
)

In [ ]:
(
    gg.ggplot(
        predictability_score_with_ops,
        gg.aes(x="tsne_x", y="tsne_y"),
    )
    + gg.geom_point(size=0.5, data=df_embeddings)
    + gg.geom_point(gg.aes(color="T4"))
)

# comparison with growth fitness screen

In [ ]:
(
    gg.ggplot(
        predictability_score.query("T4 < -3"),
        gg.aes(x="predictability_score", y="late_effect_score"),
    )
    + gg.geom_point()
    + gg.stat_smooth(method="loess", se=False)
)

In [ ]:
(
    gg.ggplot(
        predictability_score.query("T4 < -3"),
        gg.aes(x="predictability_score", y="FITNESS_PC2"),
    )
    + gg.geom_point()
    + gg.stat_smooth(method="loess", se=False)
)

In [ ]:
def compute_dosage_sensitivity(my_df):
    v1 = my_df.loc[my_df["N Mismatch"] == 0]
    v2 = my_df.loc[my_df["N Mismatch"] == 2]

    # Extract values or assign NaN if the subset is empty
    value_full = v1["Value"].iloc[0] if not v1.empty else np.nan
    value_partial = v2["Value"].iloc[0] if not v2.empty else np.nan

    # Attempt to retrieve the spacer name from any row in the group
    spacer_name = my_df["sgRNA"].iloc[0] if not my_df.empty else np.nan

    return pd.Series(
        {
            "sgRNA": spacer_name,
            "diff": value_full - value_partial,
            "value_full": value_full,
            "value_partial": value_partial,
        }
    )


# Execution
ops_dosage_sensitivity = (
    ops_df.groupby(["Estimator", "Variable(s)", "TargetID"])
    .apply(compute_dosage_sensitivity)
    .reset_index()
)

In [ ]:
ops_growth_dosage_sensitivity = ops_dosage_sensitivity.query("Estimator == 'Mean (Robust)'").loc[
    lambda x: x["Variable(s)"] == "Instantaneous Growth Rate: Volume"
]
ops_growth_dosage_sensitivity

In [ ]:
merged_df = predictability_score.merge(
    ops_growth_dosage_sensitivity,
    left_on="spacer",
    right_on="sgRNA",
    how="left",
).assign(cv=lambda x: x["diff"] / x["value_0"])
merged_df

In [ ]:
merged_df_characterized = merged_df.dropna(subset=["diff"])
merged_df_characterized

In [ ]:
(gg.ggplot(merged_df_characterized, gg.aes(x="cv", y="T4")) + gg.geom_point())

In [ ]:
merged_df_characterized["predictability_score"].plot.hist()
plt.show()

In [ ]:
(
    gg.ggplot(merged_df.query("T4 < -5"), gg.aes(x="predictability_score", y="T2"))
    + gg.geom_point()
    + gg.stat_smooth(method="loess", se=False)
)

In [ ]:
(
    gg.ggplot(merged_df.query("T4 < -5"), gg.aes(x="predictability_score", y="T4"))
    + gg.geom_point()
    + gg.stat_smooth(method="loess", se=False)
)

In [ ]:
(
    gg.ggplot(merged_df_characterized.query("T4 < -3"), gg.aes(x="predictability_score", y="cv"))
    + gg.geom_point()
    + gg.stat_smooth(method="loess", se=False)
)

In [ ]:
# Get, when possible, the TargetID associated to each sgRNA in the AnnData (accept NaN values)
# Compute each sgRNA's predictability status'
# n_guides_in_transcripts,
# ensure these are unique when not NaN

# on the OPS side, compute, for each TargetID,

In [ ]:
vis_spacers(adata, "lpxB")

In [ ]:
np.unique(list(adata.obs.loc[adata.obs["gene"] == "lpxK"]["spacer"]))

In [ ]:
plot_df = (
    ops_df.loc[lambda x: x["TargetID"] == 0]
    .query("Estimator == 'Mean (Robust)'")
    .loc[lambda x: x["Variable(s)"] == "Delta time (s)"]
)

(gg.ggplot(plot_df, gg.aes(x="N Mismatch", y="Value")) + gg.geom_point())

In [ ]:

# ---------------------------------------------------------
# Verification of Dosage Insensitive vs Sensitive Essential Genes
# ---------------------------------------------------------

# Filter for essential genes (T4 < -3)
essential_df = merged_df[merged_df["T4"] < -3].copy()
print(f"Number of essential sgRNAs (T4 < -3) with dosage data: {len(essential_df)}")

# Define sensitive vs insensitive based on dosage difference
# Sensitive: 0MM causes defect, 2MM rescues (large negative diff)
sensitive = essential_df[essential_df["diff"] < -0.2]

# Insensitive: 0MM and 2MM are similar (diff near 0)
insensitive = essential_df[(essential_df["diff"] > -0.1) & (essential_df["diff"] < 0.1)]

print(f"\nSensitive (diff < -0.2): {len(sensitive)}")
print(f"Mean value_partial (2MM growth) for sensitive: {sensitive['value_partial'].mean():.4f}")
print(f"Mean value_full (0MM growth) for sensitive: {sensitive['value_full'].mean():.4f}")

print(f"\nInsensitive (|diff| < 0.1): {len(insensitive)}")
print(f"Mean value_partial (2MM growth) for insensitive: {insensitive['value_partial'].mean():.4f}")
print(f"Mean value_full (0MM growth) for insensitive: {insensitive['value_full'].mean():.4f}")

# Check distribution of growth rates within the insensitive group
print("\nDistribution of value_full (0MM Growth) for Insensitive group:")
print(insensitive['value_full'].describe())

# Check for bimodal distribution (Low Growth vs High Growth)
# Low Growth (< 0.6): Likely hypersensitive (even 2MM is lethal)
# High Growth (>= 0.6): Likely escapers/ineffective (even 0MM is healthy)
insensitive_low = insensitive[insensitive['value_full'] < 0.6]
insensitive_high = insensitive[insensitive['value_full'] >= 0.6]

print(f"\nInsensitive Low Growth (Hypersensitive?): {len(insensitive_low)}")
if len(insensitive_low) > 0:
    print(f"Mean value_full: {insensitive_low['value_full'].mean():.4f}")

print(f"Insensitive High Growth (Escapers/Ineffective?): {len(insensitive_high)}")
if len(insensitive_high) > 0:
    print(f"Mean value_full: {insensitive_high['value_full'].mean():.4f}")

# Visualize the bimodal distribution
(
    gg.ggplot(insensitive, gg.aes(x="value_full"))
    + gg.geom_histogram(binwidth=0.05, fill="steelblue", color="white")
    + gg.labs(title="Distribution of Growth Rates (0MM) for Dosage-Insensitive Essential Genes", x="Instantaneous Growth Rate (0MM)")
)

